<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 4 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">湖表与内部表关联</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">查询湖上的订单，与 Doris 客户表关联并核对导入结果。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

本实验会准备十笔 Iceberg 湖上订单，关联 Doris 内部客户表，再核对导入结果：十行、税前金额 12220.60。

[讲义](course4_querying_external_data.md) · [课程入口](../README.md)


## 准备湖表实验

先完成 Lab 1。运行下一格将连接课程 Doris，并启动课程专用的 MinIO 对象存储与 Iceberg REST Catalog 两个辅助容器，创建十笔湖上样本。首次启动需要下载镜像，后续运行复用数据。

Doris 沿用原来的单容器。辅助服务监听本机 51900、51818 端口，使用公开的本地实验凭据。运行前确认端口空闲；停止与数据保留方式见[湖表环境说明](../../environments/lakehouse/README.md)。本实验仅重建内部表 customers_sample、orders_from_lake，湖表按实验库独立存放。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.lakehouse import prepare_lakehouse
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows, sample
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()
source = prepare_lakehouse(lab, start=True)




## 1. 直查湖表

Doris 通过 Catalog 找到 Iceberg 表，再读取它的数据文件。下一格检查表类型、查询计划与订单内容。预期为十笔订单、税前金额 12220.60；此时订单仍保存在湖表中。


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10,"12220.60")])
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1");
lab.sql(f"SELECT order_id, customer_id, order_date, order_amount FROM {source} ORDER BY order_id", title="湖上十笔订单")


## 2. 与内部客户表关联

本步骤仅重建 customers_sample 与 orders_from_lake。先建立唯一键客户表，与湖上订单按 customer_id 关联：每笔订单应恰好找到一条客户记录。

再将六个订单字段导入 orders_from_lake，逐字段比较湖表与内部表。关联和导入完成后，仍应有十笔订单、税前金额 12220.60。


In [ ]:
lab.execute("DROP TABLE IF EXISTS customers_sample")
lab.execute('CREATE TABLE customers_sample (customer_id BIGINT, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("customers_sample", ["customer_id", "customer_name"],
           [(r["customer_id"], r["customer_name"]) for r in sample()["customers"]])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN customers_sample c ON o.customer_id=c.customer_id"),
       [(10, "12220.60")])
lab.execute("DROP TABLE IF EXISTS orders_from_lake")
ddl = order_ddl("orders_from_lake")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_from_lake ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_lake ORDER BY order_id"),
       history_rows())
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_from_lake ORDER BY order_id", title="导入后的订单明细")


## 完成与边界

能够通过 Catalog 查询 Iceberg 订单表；关联内部客户后订单行数和金额保持一致；导入后的内部表逐字段匹配湖表。


## 自己动手

对比直查湖表与查询 orders_from_lake 的执行计划，找出各自的扫描对象。说明 Iceberg 表元数据在查询中的作用，以及为什么客户维表需要按 customer_id 保持唯一。


## 独立练习

用 LEFT JOIN 查询湖表第二天的订单数、金额和缺失客户数，再对比湖表与内部表的扫描计划。预期 5 笔、8276.40、缺失客户 0。

在下一格编写并运行代码，完成后再展开参考解答。空白练习不会被自动判定为完成。


In [ ]:
# 在这里编写你的 SQL 或导入请求。


<details>
<summary>参考解答（完成后再展开）</summary>

```python
query = f"""SELECT COUNT(*) AS orders, SUM(o.order_amount) AS amount,
SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers
FROM {source} o LEFT JOIN customers_sample c ON o.customer_id=c.customer_id
WHERE o.order_date='2013-01-02'"""
lab.sql(query, title="湖上第二天订单的客户匹配")
expect(lab.query(query), [(5,"8276.40",0)])
lab.sql("EXPLAIN SELECT * FROM orders_from_lake WHERE order_date='2013-01-02'", title="内部表扫描")
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_date='2013-01-02'", title="湖表扫描")
```

</details>
